In [3]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Dataset
import numpy as np
import pandas as pd
from PIL import Image

In [4]:
# Setting up the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

Training on: cuda


In [5]:
# Data augmentation for the training datasets
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

#Loading the dataset
train_dataset = ImageFolder(root=r'C:\LOCAL DISK D\Eyad\Faculty assignemts\Sem 4\Intro to AI\Project\dataset\train', transform=train_transforms)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)

print(f"Found {len(train_dataset)} training images across {len(train_dataset.classes)} classes.")

#Imbalanced number of images handler
class_counts = np.bincount(train_dataset.targets)
total_samples = len(train_dataset)
class_weights = total_samples / (len(train_dataset.classes) * class_counts)
weights_tensor = torch.FloatTensor(class_weights).to(device)

print("Class weights succesfully calculated to balance Boho and Minimalist disparity.")


Found 13163 training images across 17 classes.
Class weights succesfully calculated to balance Boho and Minimalist disparity.


In [6]:
#The CNN implementation 
class CNN(nn.Module):
    def __init__(self, num_classes = 17):
        super(CNN, self).__init__()

        #Convolutional layers
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2,2)

        self.fc1 = nn.Linear(256*14*14, 512)
        self.fc2 = nn.Linear(512, num_classes)

        # Anti-Overfitting Mechanism
        self.dropout = nn.Dropout(0.5)
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = self.pool(F.relu(self.conv4(x)))

        x = x.view(x.size(0), -1) #Flattening the pooling results

        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x
    
#Initializing the model
model = CNN(num_classes = 17).to(device)
#Making the loss function
criterion = nn.CrossEntropyLoss(weight=weights_tensor)
#Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Custom CNN succesfully built and pushed to GPU")


Custom CNN succesfully built and pushed to GPU


In [7]:
epochs = 30
print("Started Training the big boy ツ")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for i, (images, labels) in  enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()       #Clear old math
        outputs = model(images)     #Guess the styles
        loss = criterion(outputs, labels) #Check how wrong the model was
        loss.backward()             #Backtrack to error source and see how to fix it
        optimizer.step()            #Update the brain

        running_loss += loss.item()

        #Printing a little update every 100 batch to check that my laptop wasn't fried :>
        if (i+1)%100 == 0:
            print(f"  Batch {i+1}/{len(train_loader)}")
    avg_loss = running_loss/len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] completed. Average Training Loss: {avg_loss:.4f}")

print("MODEL IS NOW TRAINED HOORAY")

Started Training the big boy ツ
  Batch 100/412
  Batch 200/412
  Batch 300/412
  Batch 400/412
Epoch [1/30] completed. Average Training Loss: 2.7941
  Batch 100/412
  Batch 200/412
  Batch 300/412
  Batch 400/412
Epoch [2/30] completed. Average Training Loss: 2.6903
  Batch 100/412
  Batch 200/412
  Batch 300/412
  Batch 400/412
Epoch [3/30] completed. Average Training Loss: 2.6354
  Batch 100/412
  Batch 200/412
  Batch 300/412
  Batch 400/412
Epoch [4/30] completed. Average Training Loss: 2.5731
  Batch 100/412
  Batch 200/412
  Batch 300/412
  Batch 400/412
Epoch [5/30] completed. Average Training Loss: 2.5239
  Batch 100/412
  Batch 200/412
  Batch 300/412
  Batch 400/412
Epoch [6/30] completed. Average Training Loss: 2.4669
  Batch 100/412
  Batch 200/412
  Batch 300/412
  Batch 400/412
Epoch [7/30] completed. Average Training Loss: 2.4263
  Batch 100/412
  Batch 200/412
  Batch 300/412
  Batch 400/412
Epoch [8/30] completed. Average Training Loss: 2.3762
  Batch 100/412
  Batch 2

In [9]:
class TestDataset(Dataset):
    def __init__(self, folder_path, transform=None):
        self.folder_path = folder_path
        self.image_files = [f for f in os.listdir(folder_path) if f.endswith(('.jpg', '.jpeg', '.png'))]
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.folder_path, img_name)
        
        try:
            # Try to open the image normally
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            # IF IT FAILS: Print a warning and create a fake black image to prevent a crash
            print(f"  [!] Warning: {img_name} is corrupted. Using a blank placeholder.")
            image = Image.new('RGB', (224, 224), (0, 0, 0))
        
        if self.transform:
            image = self.transform(image)
            
        return image, img_name
    
# Setting up the testing data
test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_folder_path = r'C:\LOCAL DISK D\Eyad\Faculty assignemts\Sem 4\Intro to AI\Project\dataset\test'

test_dataset = TestDataset(folder_path=test_folder_path, transform=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

model.eval() #IMPORTANT: Turns off the Dropout constraint for maximum testing power
predictions = []

print(f"Starting inference on {len(test_dataset)} test images...")

with torch.no_grad():
    for images, img_names in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        for img_name, pred in zip(img_names, predicted):
            predictions.append({'ImageName': img_name, 'ClassLabel':pred.item()})
        if (i+1) % 25 == 0:
            print(f"  Processed batch {i+1}/{len(test_loader)}...")

#Generating final CSV file for pushing on kaggle
submission_df = pd.DataFrame(predictions)
submission_df = submission_df.sort_values(by="ImageName")

submission_df.to_csv('Eyad_model_Jackpotbaby', index = False)
print("SUCCESS, CSV is now saved in the current workspace")

Starting inference on 5482 test images...
  [!] Warning: testimage_1881.jpg is corrupted. Using a blank placeholder.
  [!] Warning: testimage_3341.jpg is corrupted. Using a blank placeholder.


c:\Users\Habib\AppData\Local\Programs\Python\Python312\Lib\site-packages\PIL\Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


  [!] Warning: testimage_3427.jpg is corrupted. Using a blank placeholder.
  [!] Warning: testimage_4180.jpg is corrupted. Using a blank placeholder.
SUCCESS, CSV is now saved in the current workspace
